# Aprovação humana e idempotência

O laboratório percorre uma proposta de chamado, a decisão do operador e a execução repetida. Cada chamada abre um processo separado e compartilha apenas o arquivo SQLite.

## Preparar uma base descartável

Abra este notebook pelo Jupyter no módulo 0011. As dependências adicionais estão no `requirements.txt`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile

pasta = Path.cwd()
if pasta.name != "0011_aprovacao_idempotencia":
    pasta = pasta.parent
arquivo_cli = pasta / "src" / "main.py"
assert arquivo_cli.is_file(), "Execute o notebook a partir deste módulo ou da pasta notebooks."

ambiente = tempfile.TemporaryDirectory()
base = Path(ambiente.name) / "lab.sqlite"

def comando(*argumentos, sucesso=True):
    retorno = subprocess.run(
        [sys.executable, str(arquivo_cli), "--base", str(base), *argumentos],
        capture_output=True,
        text=True,
    )
    if sucesso:
        retorno.check_returncode()
        return json.loads(retorno.stdout)
    assert retorno.returncode == 1, retorno
    return retorno.stderr.strip()


## Registrar o que será revisado

O operador recebe uma proposta com título e prioridade fixados. Tentar executar antes da aprovação retorna um erro.

In [ ]:
pedido = "pedido-42"
proposta = comando("propor", pedido, "--titulo", "Checkout indisponível", "--prioridade", "alta")
assert proposta["estado"] == "pendente"
print(proposta["estado"], proposta["titulo"])
print(comando("executar", pedido, sucesso=False))
assert comando("consultar", pedido)["chamado"] is None


## Aprovar e repetir a execução

A chamada pós-aprovação cria o chamado. A segunda chamada recupera o mesmo efeito.

In [ ]:
decisao = comando("decidir", pedido, "aprovar", "--operador", "plantao")
assert decisao["estado"] == "aprovada"
primeira = comando("executar", pedido)
segunda = comando("executar", pedido)
assert primeira == segunda
assert segunda["estado"] == "executada"
print("Chamado criado e recuperado:", segunda["chamado"])


## Testar a decisão de recusa

Uma proposta recusada continua sem chamado. O título de uma proposta registrada também permanece estável.

In [ ]:
comando("propor", "pedido-43", "--titulo", "Erro na entrega")
comando("decidir", "pedido-43", "rejeitar", "--operador", "plantao")
print(comando("executar", "pedido-43", sucesso=False))
print(comando("propor", pedido, "--titulo", "Título diferente", sucesso=False))
assert comando("consultar", "pedido-43")["estado"] == "rejeitada"
ambiente.cleanup()


## Limites do exemplo

A garantia acima vale para o efeito gravado no SQLite. Um sistema de chamados externo precisa reconhecer a chave de idempotência. A CLI didática recebe o nome do operador como argumento. Em produção, a aplicação obtém essa identidade da autenticação.